# 19 · Snap to street network, checked against pyproj

## What this checks

**Snap to street network** moves each site to its nearest street junction. Everything that walks the network then
starts from that junction. The widget also reports how far it moved each site. This check covers release 0.2.0.

It is the first step of a walking-time analysis. If the wrong junction is chosen, every walk after it starts in the
wrong place. So it is worth checking.

## How it is checked

A snap is a search for the nearest point, and a distance. Both have exact answers, so there is no tolerance to argue
about. A disagreement would be about the projection, or about which junction is nearest.

We repeat it with **pyproj**, the Python link to PROJ, the projection library most GIS software uses. It converts
the sites and the junctions to metres on a flat map. Then it takes the junction at the shortest straight-line
distance. No code is shared with Fieldwork.

The data are 8 pumps from the John Snow example and the 738 junctions of Soho's street network. We use eight pumps
because a search from a single site tests almost nothing.

**Ties are reported whether or not there are any.** If two junctions were exactly the same distance from a site, the
choice between them would be arbitrary. That is the one case where both sides could fairly differ. So the check also
reports, for each site, how much nearer the chosen junction is than the next one.

## What should happen

1. **The junction chosen** for each site is the same, exactly.
2. **The distance moved** agrees within 5 millimetres.
3. **The returned position** is the junction's, not the site's. That is what snapping means.
4. **The counts** of sites and of distinct junctions match exactly.
5. **No site is beyond the 150 m limit.** The widget refuses such a site, rather than snapping it.

These criteria were fixed before the check was run.

**Why 5 mm, not 1 mm.** The widget computes the distance in full, then reports it to the nearest centimetre. So 5 mm
is half of its last reported digit. It says nothing about the widget's accuracy. The first version of this check
asked for 1 mm, and failed at 3.7 mm. That was rounding and nothing else: the criterion had not read what the widget
says it reports.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_19_snap import run

report = run(Path.cwd() / "fixtures" / "snap-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["sites"], "sites against", report["networkNodes"], "nodes")
print()
print(f"{'site':<22} {'node':<10} {'moved m':>9} {'margin m':>10}")
for row in report["perSite"]:
    print(f"{row['name'][:22]:<22} {row['fieldworkNode']:<10} "
          f"{row['fieldworkDistanceM']:>9.2f} {row['marginToSecondNearestM']:>10.2f}")
print()
print(f"worst distance error {report['worstDistanceErrorMetres']:.3e} m "
      f"(reported precision {report['reportedPrecisionM']} m)")
print(f"equidistant ties {report['equidistantTies']}, "
      f"smallest margin to the second nearest {report['smallestMarginToSecondNearestM']:.2f} m")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.2.0:
**Agrees**.

- **Junctions.** All 8 sites were snapped to the same junction on both sides.
- **Distances.** The largest difference in distance moved was about 3.7 × 10⁻³ m. The widget reports to 0.01 m.
- **Positions.** Every returned position was the junction's.
- **Ties.** There were no ties. The smallest margin to the next nearest junction was 0.43 m.

Running the cell above repeats the check on the same data.

## The margin column is the interesting one

A snap is a choice between junctions. The check reports how much nearer the chosen junction is than the next one.
**One site's margin is 0.43 m.** There is no tie, so the answer is clear. But a margin that small means the choice is
decided by less than the width of a pavement. A different projection, datum or junction position would flip it.

That is reported, not hidden. A check that said only "the junctions agree" would suggest a robustness this data does
not have.

## What this does not show

- That snapping is harmless. It moves the site, by 2.2 to 20.9 m here, against a 375 m five-minute budget. Both sides
  move it in the same way. The walk is no longer free: Network isochrone has charged it against the budget since
  14 October 2026, and Reachable junctions since its release 0.4.0. This check does not test that charge; check 07
  does, for Reachable junctions.
- That the choice is robust. One of the eight sites was decided by 0.43 m.